# 03. AUV seabed surveying (Sec. VIII)

An autonomous underwater vehicle must sweep an area $A$ at fixed depth, moving at velocity $v$ with sensor field of view $r$. The constraints couple time, energy, and cost cyclically:

- Coverage: $v \cdot T \cdot r \geq k \cdot A$
- Actuation: $P_{act} \geq \psi(v)$ (drag scales as $v^3$)
- Sensing: $P_{sens} \geq \chi(r)$ (sensor power scales with $r$)
- Energy: $E \geq (P_{act} + P_{sens}) \cdot T$

Larger $v$ shortens $T$ but costs more drag; wider $r$ shortens $T$ but costs more sensor power. The MCDP solver returns the (time, energy, cost) Pareto front for each area to cover.


## Imports

In [1]:
import math
from codesign import (
    Antichain, FunctionDP, Loop, NamedProduct, Reals,
    solve, minimize_cost,
)

## Build the AUV model

In [2]:
def make_auv():
    K_GEOM = 1.0; V_MAX = 3.0; R_MAX = 5.0
    PSI_A = 30.0; CHI_A = 50.0; SENSOR_COST_A = 200.0

    Design = NamedProduct({"v": Reals(unit="m/s"), "r": Reals(unit="m")})
    F = NamedProduct({"A": Reals(unit="m^2"), "design": Design})
    R = NamedProduct({
        "design": Design,
        "T": Reals(unit="s"), "E": Reals(unit="J"), "cost": Reals(unit="$"),
    })

    def h(f):
        A = f["A"]
        v_in, r_in = f["design"]["v"], f["design"]["r"]
        if v_in == math.inf or r_in == math.inf:
            return Antichain.singleton(R, {
                "design": {"v": math.inf, "r": math.inf},
                "T": math.inf, "E": math.inf, "cost": math.inf,
            })
        v = max(float(v_in), 0.1); r = max(float(r_in), 0.5)
        if v > V_MAX or r > R_MAX:
            return Antichain.singleton(R, {
                "design": {"v": math.inf, "r": math.inf},
                "T": math.inf, "E": math.inf, "cost": math.inf,
            })

        pts = []
        for v_try in (v, min(v*1.3, V_MAX), min(v*1.7, V_MAX)):
            for r_try in (r, min(r*1.3, R_MAX), min(r*1.7, R_MAX)):
                if v_try < v_in or r_try < r_in:
                    continue
                T_try = K_GEOM * A / (v_try * r_try)
                E_try = (PSI_A * v_try**3 + CHI_A * r_try) * T_try
                cost_try = SENSOR_COST_A * r_try
                pts.append({
                    "design": {"v": v_try, "r": r_try},
                    "T": T_try, "E": E_try, "cost": cost_try,
                })
        return Antichain.from_set(R, pts)

    inner = FunctionDP(F=F, R=R, h_fn=h, name="auv_inner")
    return Loop(inner, axis="design")

auv = make_auv()
auv

DP(loop(auv_inner, axis=design): A:R+[m^2] -> A[T:R+[s]×E:R+[J]×cost:R+[$]])

## Run for three mission scales

Each area gives a small (T, E, cost) Pareto front. We then collapse it to a single design with `minimize_cost` using a composite scalar objective.


In [3]:
def show(result, label):
    print(label)
    print(f"   iters={result.iterations}, feasible={result.feasible}")
    if not result.feasible:
        return
    for p in result.antichain.points:
        print(f"   T={p['T']:.0f}s, E={p['E']/1000:.1f}kJ, $={p['cost']:.0f}")
    best = minimize_cost(
        result,
        cost_fn=lambda r: r["T"] + 0.05 * (r["E"] / 1000.0) + r["cost"],
    )
    if best is not None:
        print(f"   best composite: T={best['T']:.0f}s, "
              f"E={best['E']/1000:.1f}kJ, $={best['cost']:.0f}")
    print()

for A in (100.0, 1000.0, 10_000.0):
    result = solve(auv, {"A": A}, max_iter=50)
    show(result, f"Area = {A:g} m^2")

Area = 100 m^2
   iters=2, feasible=True
   T=1176s, E=29.6kJ, $=100
   T=905s, E=29.5kJ, $=130
   T=692s, E=29.5kJ, $=170
   best composite: T=692s, E=29.5kJ, $=170

Area = 1000 m^2
   iters=2, feasible=True
   T=11765s, E=295.9kJ, $=100
   T=9050s, E=295.5kJ, $=130
   T=6920s, E=295.1kJ, $=170
   best composite: T=6920s, E=295.1kJ, $=170

Area = 10000 m^2
   iters=2, feasible=True
   T=117647s, E=2958.5kJ, $=100
   T=90498s, E=2954.5kJ, $=130
   T=69204s, E=2951.4kJ, $=170
   best composite: T=69204s, E=2951.4kJ, $=170



## Interpretation

Each scenario produces three incomparable points along the (cost, speed) tradeoff: slow-and-cheap or fast-and-expensive sensors. As $A$ grows by a factor of 10, both $T$ and $E$ scale up by the same factor (linear in area), while cost stays the same (it's a one-off sensor purchase, not a per-mission cost).
